# PatchTST-style

Nie, Nguyen, Sinthong, Kalagnanam, "A Time Series is Worth 64 Words: Long-term Forecasting with Transformers", ICLR 2023 ([arXiv:2211.14730](https://arxiv.org/abs/2211.14730)).

Patches a univariate series into tokens (like ViT patchifies an image, but 1D) and forecasts each channel of real ETTh1 with the SAME weights, independently (channel independence).

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from transformer_playground.data import load_etth1
from transformer_playground.device import resolve_device
from model import PatchTSTModel

device = resolve_device('auto')
print('device:', device)

In [ ]:
raw = load_etth1()
n_total = raw.shape[0]
n_train = int(n_total * 0.7)
n_val = int(n_total * 0.1)
train_raw = raw[:n_train]
test_raw = raw[n_train + n_val:]
mean, std = train_raw.mean(axis=0, keepdims=True), train_raw.std(axis=0, keepdims=True)
train_norm = (train_raw - mean) / std
test_norm = (test_raw - mean) / std

seq_len, pred_len = 96, 24
def make_windows(data, seq_len, pred_len):
    import numpy as np
    T, C = data.shape
    n = T - seq_len - pred_len + 1
    xs = np.stack([data[i:i+seq_len] for i in range(n)])
    ys = np.stack([data[i+seq_len:i+seq_len+pred_len] for i in range(n)])
    return torch.from_numpy(xs).permute(0,2,1).float(), torch.from_numpy(ys).permute(0,2,1).float()

x_train, y_train = make_windows(train_norm, seq_len, pred_len)
x_test, y_test = make_windows(test_norm, seq_len, pred_len)
print(f'{n_train} train hours, {len(test_norm)} test hours, 7 channels, {x_train.shape[0]} train windows')

In [ ]:
model = PatchTSTModel(seq_len=seq_len, pred_len=pred_len).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

batch_size = 32
n = x_train.shape[0]
history = {'train_mse': []}
for epoch in range(5):
    perm = torch.randperm(n)
    for i in range(0, n, batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = x_train[idx].to(device), y_train[idx].to(device)
        opt.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward(); opt.step()
    history['train_mse'].append(loss.item())
    print(f'epoch {epoch} | train_mse {loss.item():.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(x_test[:1].to(device))[0, -1].cpu()  # last channel (OT), first test window
true = y_test[0, -1]
plt.plot(true, label='true')
plt.plot(preds, label='predicted')
plt.xlabel('forecast horizon step')
plt.ylabel('normalized OT')
plt.legend()
plt.title('PatchTST-style forecast, real ETTh1 (oil temperature channel)')
plt.show()